# 0.Library

In [1]:
import pandas as pd
from pathlib import Path
import re

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)

<module 'features.general_func' from 'd:\\my-github\\detection_AD_with_VR_data\\src\\notebooks\\..\\features\\general_func.py'>

# 1. Constants and paths

In [2]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
data_folder   = parent_folder / "data"
input_path_of_json   = data_folder / "patients_data_log.json"
patients_data_log_df = pd.read_json(input_path_of_json)

patient_ids          = patients_data_log_df["PatientID"].to_list()
patients_data_folder = data_folder / "PatientsData"

# empty dictionaries
patient_dict = {
    "Paitent_id"    : None,
    "Age"           : None,
    "Gender"        : None,
    "dominant_hand" : None,
    "Sessions_Completed_out_of_4" : None, 
    "Help_Rating_out_of_5" : None
}

phase_dict = {
    "phase_id"       : None,
    "phase_name"     : None,
    "phase_duration" : None
}

# statistical features
reading_time_dict = {
    "total_reading_time": None,
    "mean_reading_time": None,
    "max_reading_time": None,
    "median_reading_time": None,
    "std_reading_time": None,
    "intensity_reading_time" : None
}

hover_dict = {
    "total_count_hover": None,
    "total_duration_hover": None,
    "mean_duration_hover": None,
    "max_duration_hover": None,
    "median_duration_hover": None,
    "std_duration_hover": None,
    "intensity_hover": None
}

press_dict = {
    "total_count_press": None,
    "total_duration_press": None,
    "mean_duration_press": None,
    "max_duration_press": None,
    "median_duration_press": None,
    "std_duration_press": None,
    "intensity_press": None # Interaction intensity
}

grab_dict = {
    "total_count_grab": None,
    "total_duration_grab": None,
    "mean_duration_grab": None,
    "max_duration_grab": None,
    "median_duration_grab": None,
    "std_duration_grab": None,
    "intensity_grab": None # Interaction intensity
}

gaze_dict = {
    "total_count_gaze": None,
    "total_duration_gaze": None,
    "mean_duration_gaze": None,
    "max_duration_gaze": None,
    "median_duration_gaze": None,
    "std_duration_gaze": None,
    "intensity_gaze": None # Interaction intensity
}

# Behavioral features
behavior_dict = {
    "hover_vs_reading_time_ratio": None,
    "hover_vs_active_interaction_ratio": None,
    "interaction_fraction": None,
    "decision_latency": None,  
    "clicks_per_second": None,
    "hovers_per_click": None
}

# Temporal Features
temporal_dict = {
    "time_before_first_press": None,
    "time_before_first_hover": None
}

# Head features
headset_dict = {
    # Position features
    "head_total_distance": None,

    "HMD_X_std": None,
    "HMD_Y_std": None,
    "HMD_Z_std": None,

    "HMD_X_range": None,
    "HMD_Y_range": None,
    "HMD_Z_range": None,

    # Speed features in ditance
    "mean_head_speed_in_distance": None,
    "max_head_speed_in_distance": None,
    "std_head_speed_in_distance": None,

    # Orientation features
    "head_total_orientation" : None,

    "Yaw_std": None,
    "Pitch_std": None,
    "Roll_std": None,

    "Yaw_range": None,
    "Pitch_range": None,
    "Roll_range": None,

    # Speed features in Orientation
    "mean_head_speed_in_orientation": None,
    "max_head_speed_in_orientation": None,
    "std_head_speed_in_orientation": None,

    # Gaze features
    "gaze_obj_looked_ratio": None,
    "gaze_switch_count": None

}

# Hand features
controller_dict = {
    # Distance features
    "dominant_hand_total_distance": None,
    "not_dominant_hand_total_distance": None,

    # Speed features
    "dominant_hand_mean_speed": None,
    "not_dominant_hand_mean_speed": None,
    "dominant_hand_max_speed": None,
    "not_dominant_hand_max_speed": None,

    # Workspace_volume features
    "dominant_hand_x_range": None,
    "dominant_hand_y_range": None,
    "dominant_hand_z_range": None,
    "not_dominant_hand_x_range": None,
    "not_dominant_hand_y_range": None,
    "not_dominant_hand_z_range": None,

    # Press features
    "dominant_hand_trigger_press_count": None,
    "not_dominant_hand_trigger_press_count": None,
    "dominant_hand_trigger_pressure_mean": None,
    "not_dominant_hand_trigger_pressure_mean": None,

    # Grip count features
    "dominant_hand_grip_count": None,
    "not_dominant_hand_grip_count": None,
    "dominant_hand_grip_mean": None,
    "not_dominant_hand_grip_mean": None,
}

phase_name_list    = ['Tutorial', 'ObjectRecognition', 'Visuospatial', 'Memory']
event_empty_dictionaries = [reading_time_dict, hover_dict, press_dict, grab_dict, gaze_dict, behavior_dict, temporal_dict]
trajectory_empty_dictionaries = [headset_dict, controller_dict]
all_four_phase_dict = []

In [3]:
x = 0
for i in event_empty_dictionaries:
    x += len(i.keys())

for i in trajectory_empty_dictionaries:
    x += len(i.keys())

x += len(phase_dict.keys())
x

87

In [4]:
for i in trajectory_empty_dictionaries:
    print(len(i.keys()))

22
20


In [5]:
all_keys = []

all_keys = list(patient_dict.keys()) + list(phase_dict.keys())

for empty_dictionary in event_empty_dictionaries:
    columns = empty_dictionary.keys()
    all_keys += list(columns)

for empty_dictionary in trajectory_empty_dictionaries:
    columns = empty_dictionary.keys()
    all_keys += list(columns)

all_keys

['Paitent_id',
 'Age',
 'Gender',
 'dominant_hand',
 'Sessions_Completed_out_of_4',
 'Help_Rating_out_of_5',
 'phase_id',
 'phase_name',
 'phase_duration',
 'total_reading_time',
 'mean_reading_time',
 'max_reading_time',
 'median_reading_time',
 'std_reading_time',
 'intensity_reading_time',
 'total_count_hover',
 'total_duration_hover',
 'mean_duration_hover',
 'max_duration_hover',
 'median_duration_hover',
 'std_duration_hover',
 'intensity_hover',
 'total_count_press',
 'total_duration_press',
 'mean_duration_press',
 'max_duration_press',
 'median_duration_press',
 'std_duration_press',
 'intensity_press',
 'total_count_grab',
 'total_duration_grab',
 'mean_duration_grab',
 'max_duration_grab',
 'median_duration_grab',
 'std_duration_grab',
 'intensity_grab',
 'total_count_gaze',
 'total_duration_gaze',
 'mean_duration_gaze',
 'max_duration_gaze',
 'median_duration_gaze',
 'std_duration_gaze',
 'intensity_gaze',
 'hover_vs_reading_time_ratio',
 'hover_vs_active_interaction_ratio'

In [6]:
len(all_keys)

93

In [7]:
columns_df = patient_dict.keys()

df = pd.DataFrame(columns=columns_df)
df

,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5


# 2. Main_process

In [8]:
def creating_event_dictionary(df: pd.DataFrame, phase_name: str, empty_dictionaries:list, phase_dict: dict) -> dict:
    
    # filling read_time dict
    print(empty_dictionaries[0])
    reading_time_dict = du.filling_reading_time_dict(df, empty_dictionaries[0], phase_dict['phase_duration'])
    print(empty_dictionaries[0])
    # filling hover_dict
    hover_dict = du.filling_hover_dict(df, empty_dictionaries[1], phase_dict['phase_duration'])

    # filling press_dict
    press_dict = du.filling_press_dict(df, empty_dictionaries[2], phase_dict['phase_duration'])

    # filling grab_dict
    grab_dict = du.filling_grab_dict(df, empty_dictionaries[3], phase_dict['phase_duration'])

    # filling gaze_dict
    gaze_dict = du.filling_gaze_dict(df, empty_dictionaries[4], phase_dict['phase_duration'])

    # filling behavior_dict
    behavior_dict = du.filling_behavior_dict(df, empty_dictionaries[5], hover_dict, press_dict, reading_time_dict, phase_dict['phase_duration'])

    # filling temporal_dict
    temporal_dict = du.filling_temporal_dict(df, empty_dictionaries[6])

    # concate all dictionaries
    event_dictionary = reading_time_dict | hover_dict | press_dict | grab_dict | gaze_dict | behavior_dict | temporal_dict

    # check for extra features
    match phase_name:

        case 'Tutorial':
            return event_dictionary
        
        case 'ObjectRecognition':

            # extra features only for ObjectRecognition
            obj_recognition_dict = {
                "obj_recognition_score": None,
                "obj_recognition_mean_success_duration" : None,
                "obj_recognition_mean_choose_wrong_obj": None,
                "obj_recognition_total_duration": None
            }

            # fill and add extra features to basic dict
            obj_recognition_dict = du.filling_obj_recognition_dict(df, obj_recognition_dict)
            event_dictionary.update(obj_recognition_dict)

            return event_dictionary
        
        case 'Visuospatial':
            
            # extra features only for Visuospatial
            visuospatial_dict = {
                "visuospatial_score": None,
                "visuospatial_total_wrong_placement": None,
                "visuospatial_total_duration" : None
            }

            # fill and add extra features to basic dict
            visuospatial_dict = du.filling_visuospatial_dict(df, visuospatial_dict)
            event_dictionary.update(visuospatial_dict)
            
            return event_dictionary
        
        case 'Memory':

            # extra features only for memory_dict
            memory_dict = {
                "memory_score": None,
                "memory_delay_first_action": None,
                "memory_total_wrong_recall" : None,
                "memory_mean_recall": None,
                "memory_cognitive_freez_count": None,
                "memory_cognitive_freez_mean_duration": None,
                "memory_total_duration" : None
            }

            # fill and add extra features to basic dict
            memory_dict = du.filling_memory_dict(df, memory_dict)
            event_dictionary.update(memory_dict)
            
            return event_dictionary
        
        case _ :
            gf.fail(msg="not valid value for test_name!!!", error="ValueError")

def creating_trajectory_dictionary(df: pd.DataFrame, dominant_hand: str, not_dominant_hand: str, empty_dictionaries:list, phase_dict: dict) -> dict:

    # fill headset dictionary
    headset_dict = du.filling_headset_dict(df, empty_dictionaries[0])

    # fill controler dictionary
    controler_dict = du.filling_controller_dict(df, dominant_hand, not_dominant_hand, empty_dictionaries[0])

    # concate all dictionaries
    trajectory_dictionary = headset_dict | controler_dict

    # check dict
    gf.general_dict_check(trajectory_dictionary, "trajectory_dictionary", None)
    
    return trajectory_dictionary


In [9]:
# first loop
final_df = pd.DataFrame()
all_rows = []

for patient_id in patient_ids:
    print(patient_id)
    print("============================================================")
    # extract data path
    patient_folder_path = du.find_patient_folder(patients_data_folder=patients_data_folder,
                                                 patient_id=patient_id)
    row_index = patient_ids.index(patient_id)

    # filling patient dictionary
    patient_dict = du.filling_patient_dict(patients_data_log_df, patient_dict, row_index)

    # filling phase dictionaries
    path_to_check  = patient_folder_path / "clean_data"
    csv_files_list = du.find_csv_file(patient_folder_path / "clean_data")

    # set dominant and not dominant hand
    dominant_hand = patient_dict["dominant_hand"]
    not_dominant_hand = 'Left' if dominant_hand == 'Right' else 'Right'

    # phase id (range 1 to 4)
    phase_id = 1

    for i in range(0,8,2):

        # initial phase dict
        phase_dict["phase_id"] = phase_id

        # Event file
        csv_file_event = csv_files_list[i]
        df_event       = pd.read_csv(path_to_check / csv_file_event)

        # extract phase duration
        phase_dict["phase_duration"] = du.extract_phase_duration(df_event)

        # extract phase name
        match      = re.search(r'_([^_]+)_events\.csv$', csv_file_event)
        phase_name =  match.group(1) if match else sys.exit("phase name not found!!!")

        # filling
        phase_dict["phase_name"] = phase_name if phase_name in phase_name_list else gf.fail(msg="phase_name is found but it's invalid!!!", error="Key not found")

        print(phase_dict["phase_name"])
        event_dictionary = creating_event_dictionary(df_event, phase_name , event_empty_dictionaries, phase_dict)

        # Trajectory file
        csv_file_trajectory = csv_files_list[i+1]
        df_trajectory       = pd.read_csv(path_to_check / csv_file_trajectory)

        print(len(trajectory_empty_dictionaries))
        trajectory_dictionary = creating_trajectory_dictionary(df_trajectory, dominant_hand, not_dominant_hand, trajectory_empty_dictionaries, phase_dict)
        
        # concate phase dictionaries
        concated_phase_dict = phase_dict | event_dictionary | trajectory_dictionary
    
        all_four_phase_dict.append(concated_phase_dict)
        
        phase_id +=1

    # concate phase dictionary and patient
    
    single_row_of_df = patient_dict | all_four_phase_dict[0] | all_four_phase_dict[1] | all_four_phase_dict[2] | all_four_phase_dict[3]
    
    # add single row to final dataframe
    #final_df.loc[len(final_df)] = single_row_of_df
    all_rows.append(single_row_of_df)


P001
Tutorial
{'total_reading_time': None, 'mean_reading_time': None, 'max_reading_time': None, 'median_reading_time': None, 'std_reading_time': None, 'intensity_reading_time': None}
{'total_reading_time': np.float64(45.03), 'mean_reading_time': np.float64(15.01), 'max_reading_time': np.float64(21.0), 'median_reading_time': np.float64(17.23), 'std_reading_time': np.float64(7.36), 'intensity_reading_time': np.float64(0.92)}
2
ObjectRecognition
{'total_reading_time': np.float64(45.03), 'mean_reading_time': np.float64(15.01), 'max_reading_time': np.float64(21.0), 'median_reading_time': np.float64(17.23), 'std_reading_time': np.float64(7.36), 'intensity_reading_time': np.float64(0.92)}
{'total_reading_time': np.float64(9.0), 'mean_reading_time': np.float64(3.0), 'max_reading_time': np.float64(3.0), 'median_reading_time': np.float64(3.0), 'std_reading_time': np.float64(0.0), 'intensity_reading_time': np.float64(0.19)}
2
Visuospatial
{'total_reading_time': np.float64(9.0), 'mean_reading_time

In [ ]:
df = pd.DataFrame([single_row_of_df])
df


,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5,phase_id,phase_name,phase_duration,total_reading_time,...,visuospatial_score,visuospatial_total_wrong_placement,visuospatial_total_duration,memory_score,memory_delay_first_action,memory_total_wrong_recall,memory_mean_recall,memory_cognitive_freez_count,memory_cognitive_freez_mean_duration,memory_total_duration
0,P001,None,Female,Right,4,1,4,Memory,344.57,317.74,...,1.0,1,37.7,1.0,15.21,7,125.43,2,1.51,289.4


In [ ]:
df.columns

Index(['Paitent_id', 'Age', 'Gender', 'dominant_hand',
       'Sessions_Completed_out_of_4', 'Help_Rating_out_of_5', 'phase_id',
       'phase_name', 'phase_duration', 'total_reading_time',
       ...
       'visuospatial_score', 'visuospatial_total_wrong_placement',
       'visuospatial_total_duration', 'memory_score',
       'memory_delay_first_action', 'memory_total_wrong_recall',
       'memory_mean_recall', 'memory_cognitive_freez_count',
       'memory_cognitive_freez_mean_duration', 'memory_total_duration'],
      dtype='object', length=107)